# 02. Clasificarea imaginilor ImageNet cu DoReFaNet

Clasificăm un exemplu etichetat din ImageNet folosind modelul **QNN-MO-PYNQ** instalat. Prima convoluție și straturile complet conectate rulează pe **procesorul ARM**, iar straturile intermediare cuantizate sunt executate pe **FPGA**, cu ponderi pe 1 bit și activări pe 2 biți.

![Topologia DoReFaNet](../dorefanet-topology.svg)

Executați celulele în ordine, folosind **Shift + Enter**. Închideți mai întâi celelalte kerneluri Jupyter care folosesc FPGA. Încărcarea ponderilor mari ale straturilor complet conectate poate necesita mai multă memorie decât cei 512 MB disponibili pe PYNQ-Z2.

**Important:** înainte de încărcarea DoReFaNet pe PYNQ-Z2, creați și activați fișierul swap de 1 GiB conform instrucțiunilor de mai jos.

Începeți cu un kernel nou. Această versiune detectează automat pachetul Python instalat și folosește modelele și imaginile existente; nu este necesară reinstalarea pachetului sau descărcarea modelului. Importurile `Xlnk` neutilizate trebuie să fi fost deja eliminate din `qnn/dorefanet.py` și `qnn/tinieryolo.py`, conform documentației repository-ului.

*Traducere didactică a laboratorului; codul de inferență este preluat din versiunea engleză. Nu a fost validată separat execuția hardware a acestei copii.*

## 1. Importul bibliotecilor și identificarea fișierelor

Executați celula următoare. Aceasta importă bibliotecile, verifică existența modelului și a imaginilor și afișează directoarele utilizate. **Nu inițializează încă FPGA-ul.**

Notebook-ul rulează în mod obișnuit din directorul QNN instalat. Este verificat și directorul părinte, apoi locația standard a plăcii. Dacă este necesar, setați `NOTEBOOK_DIR` la directorul care conține `imagenet-classes.pkl` și `imagenet-samples`.

In [ ]:
import os
import gc
import pickle
import random
import time
from pathlib import Path

import numpy as np
from matplotlib import pyplot as plt
from matplotlib.ticker import PercentFormatter
from PIL import Image
from IPython.display import display

import qnn
from qnn import Dorefanet, utils

QNN_ROOT = Path(qnn.__file__).resolve().parent
QNN_PARAM_DIR = QNN_ROOT / "params"
NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "imagenet-classes.pkl").is_file():
    if (NOTEBOOK_DIR.parent / "imagenet-classes.pkl").is_file():
        NOTEBOOK_DIR = NOTEBOOK_DIR.parent
    else:
        NOTEBOOK_DIR = Path("/home/xilinx/jupyter_notebooks/qnn")
SAMPLE_DIR = NOTEBOOK_DIR / "imagenet-samples"
LABEL_FILE = NOTEBOOK_DIR / "imagenet-classes.pkl"

required_files = [QNN_PARAM_DIR / name for name in (
    "dorefanet-layers.json", "dorefanet-conv0.npy",
    "dorefanet-fc-normalized.npy", "ilsvrc_pp_mean.npy")]
required_files.append(LABEL_FILE)
missing = [str(path) for path in required_files if not path.is_file()]
if missing:
    raise FileNotFoundError("Required files missing:\n" + "\n".join(missing))
if not SAMPLE_DIR.is_dir():
    raise FileNotFoundError("Sample folder missing: " + str(SAMPLE_DIR))
image_files = sorted(path for path in SAMPLE_DIR.iterdir()
                     if path.is_file() and path.suffix.lower() in
                     (".jpg", ".jpeg", ".png", ".bmp"))
if not image_files:
    raise FileNotFoundError("No sample images found in " + str(SAMPLE_DIR))
print("Python package:", QNN_ROOT)
print("Notebook data:", NOTEBOOK_DIR)
print("Sample images:", len(image_files))

## 2. Pregătirea obligatorie a memoriei pe PYNQ-Z2

PYNQ-Z2 are doar **512 MB RAM**. Încărcarea ponderilor complet conectate ale DoReFaNet poate epuiza memoria disponibilă și poate reporni kernelul Jupyter. Prin urmare, înaintea primei încărcări a modelului, creați și activați un **fișier swap de 1 GiB**.

Deschideți un terminal pe PYNQ-Z2 și executați:

```bash
sudo dd if=/dev/zero of=/home/xilinx/qnn-install.swap bs=1M count=1024 status=progress
sudo chmod 600 /home/xilinx/qnn-install.swap
sudo mkswap /home/xilinx/qnn-install.swap
sudo swapon /home/xilinx/qnn-install.swap
```

Verificați activarea swap-ului:

```bash
free -h
swapon --show
```

Comanda `free -h` ar trebui să indice aproximativ **1 GiB swap**.

### După repornirea plăcii

Fișierul swap rămâne pe cardul SD, însă nu este activat automat. **Nu îl recreați.** Reactivați fișierul existent cu:

```bash
sudo swapon /home/xilinx/qnn-install.swap
```

Apoi verificați din nou folosind:

```bash
free -h
```

### Dacă kernelul Jupyter se oprește în timpul încărcării ponderilor

Un simptom tipic este mesajul **The kernel appears to have died. It will restart automatically.**, afișat după:

```text
Loading fully connected weights; please wait...
```

și înainte de mesajul `Initializing the FPGA...`. Închideți celelalte kerneluri Jupyter, verificați `swapon --show`, activați swap-ul existent dacă este nevoie, reporniți kernelul notebook-ului și reluați execuția de la început.

## 3. Încărcarea ponderilor și etichetelor, apoi inițializarea modelului

Executați celula următoare și așteptați mesajul **Model ready**. Sunt încărcate dicționarele NumPy cu ponderile existente și cele 1.000 de etichete ImageNet, după care este inițializat acceleratorul FPGA. Se folosește configurația implicită a rețelei din pachetul instalat.

Un mesaj anunță fiecare încărcare de date voluminoase. Dacă celula eșuează, nu continuați cu inferența. La reexecutare, clasificatorul inițializat anterior este eliberat înainte de crearea altuia. Fișierele pickle și NumPy sunt considerate de încredere deoarece provin din pachetul utilizat.

In [ ]:
if "classifier" in globals():
    classifier.deinit_accelerator()
    del classifier
# Avoid holding old weight dictionaries while reloading this cell.
for variable in ("conv0_weights", "fc_weights"):
    globals().pop(variable, None)
gc.collect()

print("Loading first-convolution weights...", flush=True)
conv0_weights = np.load(str(QNN_PARAM_DIR / "dorefanet-conv0.npy"),
                       encoding="latin1", allow_pickle=True).item()
print("Loading fully connected weights; please wait...", flush=True)
fc_weights = np.load(str(QNN_PARAM_DIR / "dorefanet-fc-normalized.npy"),
                    encoding="latin1", allow_pickle=True).item()
for key in ("conv0/W", "conv0/T"):
    if key not in conv0_weights:
        raise KeyError("Missing weight: " + key)
for key in ("fc0/Wn", "fc0/bn", "fc1/Wn", "fc1/bn", "fct/W"):
    if key not in fc_weights:
        raise KeyError("Missing weight: " + key)

with LABEL_FILE.open("rb") as label_stream:
    classes = pickle.load(label_stream, encoding="latin1")
if set(classes.keys()) != set(range(1000)):
    raise ValueError("Expected ImageNet labels indexed from 0 to 999.")
names = {index: classes[index][1].split(",")[0] for index in classes}
synset_to_index = {classes[index][0]: index for index in classes}

print("Initializing the FPGA...", flush=True)
classifier = Dorefanet()
try:
    net = classifier.load_network()
    classifier.init_accelerator()
except BaseException:
    classifier.deinit_accelerator()
    raise
print("Model ready. ImageNet classes:", len(classes))

## 4. Selectarea și afișarea unei imagini etichetate

Păstrați `IMAGE_NAME = None` pentru alegerea aleatorie a unei imagini. Pentru un exemplu anume, introduceți numele unui fișier existent în `imagenet-samples`. Celula afișează imaginea și clasa cunoscută și pregătește intrarea modelului.

Pentru altă imagine, reexecutați această celulă și secțiunile **5–9**. Nu este necesară reîncărcarea ponderilor din secțiunea 3. Măsurătorile precedente sunt șterse.

In [ ]:
IMAGE_NAME = None  # Example: replace None with an existing filename string.
image_path = random.choice(image_files) if IMAGE_NAME is None else SAMPLE_DIR / IMAGE_NAME
if not image_path.is_file():
    raise FileNotFoundError("Image not found: " + str(image_path))
img, img_class = classifier.load_image(str(image_path))
expected_index = synset_to_index.get(img_class)
print("Image:", image_path.name)
print("Correct class:", names[expected_index] if expected_index is not None
      else "not available from this filename")
with Image.open(str(image_path)) as opened:
    preview = opened.convert("RGB")
preview.thumbnail((480, 360))
display(preview)
stage_seconds = {}

## 5. Prima convoluție pe procesorul ARM

Executați celula următoare. Stratul utilizează ponderile originale în precizie completă. Ieșirea sa este pragată în reprezentarea cerută de straturile cuantizate pe FPGA. Timpul afișat include convoluția și pragarea.

In [ ]:
start = time.perf_counter()
conv0 = utils.conv_layer(img, conv0_weights["conv0/W"], stride=4)
conv0 = utils.threshold(conv0, conv0_weights["conv0/T"])
stage_seconds["First convolution + threshold (CPU)"] = time.perf_counter() - start
print("CPU first stage: {:.3f} s".format(
    stage_seconds["First convolution + threshold (CPU)"]))

## 6. Straturile cuantizate pe FPGA

Executați celula următoare. Același accelerator parcurge secvența configurată de straturi cuantizate. Pregătirea bufferelor și conversia ieșirii nu intră în timpul măsurat al apelului de inferență.

Măsurarea caracterizează **apelul de offload**, inclusiv costurile runtime, nu doar ciclurile de calcul ale logicii FPGA.

In [ ]:
out_dim = net["merge4"]["output_dim"]
out_ch = net["merge4"]["output_channels"]
conv_output = classifier.get_accel_buffer(out_ch, out_dim)
conv_input = np.ascontiguousarray(classifier.prepare_buffer(conv0))
start = time.perf_counter()
classifier.inference(conv_input, conv_output)
stage_seconds["Quantized-layer offload (FPGA)"] = time.perf_counter() - start
conv_output = classifier.postprocess_buffer(conv_output)
if not np.all(np.isfinite(conv_output)):
    raise ValueError("Non-finite accelerator output. Check model initialization.")
print("FPGA offload call: {:.3f} s".format(
    stage_seconds["Quantized-layer offload (FPGA)"]))

## 7. Straturile complet conectate pe procesorul ARM

Executați celula următoare. Sunt păstrate normalizarea și calculele straturilor complet conectate ale rețelei originale. O ieșire FPGA formată exclusiv din zerouri este semnalată explicit, evitând propagarea unor rezultate invalide prin împărțire la zero.

In [ ]:
scale = np.max(conv_output)
if scale <= 0:
    raise RuntimeError("Accelerator output is all zero. Stop and check initialization before continuing.")
fc_input = conv_output / scale
if fc_input.size != fc_weights["fc0/Wn"].shape[0]:
    raise ValueError("Accelerator output size does not match the fully connected weights.")
start = time.perf_counter()
fc0_out = utils.fully_connected(fc_input, fc_weights["fc0/Wn"], fc_weights["fc0/bn"])
fc0_out = utils.quantize(utils.qrelu(fc0_out), 2)
fc1_out = utils.fully_connected(fc0_out, fc_weights["fc1/Wn"], fc_weights["fc1/bn"])
fc1_out = utils.qrelu(fc1_out)
fct_b = np.zeros((fc_weights["fct/W"].shape[1],))
fct_out = utils.fully_connected(fc1_out, fc_weights["fct/W"], fct_b)
stage_seconds["Fully connected layers (CPU)"] = time.perf_counter() - start
print("CPU fully connected stages: {:.3f} s".format(
    stage_seconds["Fully connected layers (CPU)"]))

## 8. Interpretarea rezultatului clasificării

Executați celula următoare. Sunt afișate **cele cinci clase cu cele mai mari scoruri**, valorile lor softmax și verificarea Top-1 și Top-5 în raport cu eticheta cunoscută.

Scorurile sunt ieșiri ale modelului, nu garanții ale corectitudinii. Reușita pe o singură imagine nu reprezintă acuratețea întregului set de date.

In [ ]:
if np.asarray(fct_out).shape != (len(classes),) or not np.all(np.isfinite(fct_out)):
    raise ValueError("Expected 1,000 finite classification scores.")
out = utils.softmax(fct_out)
topn = utils.get_topn_indexes(out, 5)
print("Correct class:", names[expected_index] if expected_index is not None else "unknown")
for index in topn:
    print("{:25s} {:7.2%}".format(names[int(index)], float(out[index])))
if expected_index is not None:
    print("Top-1:", "correct" if int(topn[0]) == expected_index else "wrong")
    print("Top-5:", "correct class included" if expected_index in topn
          else "correct class not included")
fig, ax = plt.subplots(figsize=(8, 3.5))
positions = np.arange(len(topn))
colors = ["green" if int(index) == expected_index else "steelblue" for index in topn]
ax.barh(positions, out[topn], color=colors)
ax.set_yticks(positions)
ax.set_yticklabels([names[int(index)] for index in topn])
ax.invert_yaxis()
ax.set_xlim(0, min(1.0, float(np.max(out[topn])) * 1.25))
ax.xaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_xlabel("Softmax score (axis scaled to the displayed predictions)")
ax.set_title("Top-5 predictions (green = known correct class)")
ax.grid(axis="x", alpha=0.3)
fig.tight_layout()
plt.show()

## 9. Compararea timpilor măsurați pe etape

Executați celula următoare. Sunt prezentate cele trei etape de calcul cronometrate și suma duratelor. Etapele CPU și FPGA execută **straturi diferite**; aceasta este o împărțire a sarcinii, nu o măsurare a accelerării FPGA față de CPU pentru aceeași operație.

Suma exclude încărcarea imaginii, pregătirea și conversia bufferelor, softmax, graficele și timpii de așteptare dintre celulele notebook-ului. Nu trebuie prezentată drept FPS al camerei sau throughput al aplicației complete. Nu este salvat un fișier de temporizare comun cu alte laboratoare.

In [ ]:
stage_order = ["First convolution + threshold (CPU)",
               "Quantized-layer offload (FPGA)",
               "Fully connected layers (CPU)"]
if any(name not in stage_seconds for name in stage_order):
    raise RuntimeError("Run all three computation stages for the current image first.")
values = [stage_seconds[name] for name in stage_order]
for name, seconds in zip(stage_order, values):
    print("{}: {:.3f} s".format(name, seconds))
print("Sum of measured stages: {:.3f} s".format(sum(values)))
fig, ax = plt.subplots(figsize=(9, 3.5))
positions = np.arange(len(stage_order))
ax.barh(positions, values, color=["steelblue", "darkorange", "steelblue"])
ax.set_yticks(positions)
ax.set_yticklabels(stage_order)
ax.invert_yaxis()
ax.set_xlabel("Measured stage time (seconds)")
ax.set_title("CPU and FPGA stages of the same inference")
ax.grid(axis="x", alpha=0.3)
fig.tight_layout()
plt.show()

## 10. Finalizare și eliberarea resurselor

Executați celula următoare după încheierea testelor. Se folosește metoda de eliberare a resurselor oferită de model și sunt eliminate tablourile intermediare și dicționarele de ponderi voluminoase. Nu se utilizează `Xlnk` sau resetarea globală a dispozitivului.

Pentru o nouă rulare după eliberarea resurselor, reluați secțiunea 3. Opriți acest kernel înainte de a deschide alt notebook care utilizează FPGA.

In [ ]:
if "classifier" in globals():
    classifier.deinit_accelerator()
    del classifier
for variable in ("conv0_weights", "fc_weights", "img", "conv0", "conv_input",
                 "conv_output", "fc_input", "fc0_out", "fc1_out", "fct_out"):
    globals().pop(variable, None)
gc.collect()
print("Classifier released and large Python arrays cleared.")

**Validare:** identificarea fișierelor, încărcarea ponderilor și etichetelor, fluxul Python, graficele, reinițializarea și eliberarea resurselor au fost verificate off-board, cu accelerator simulat. Operațiile numerice ale rețelei sunt cele ale exemplului original. Bibliotecile native, execuția pe FPGA și predicțiile ImageNet necesită în continuare validare pe placa PYNQ.